# Решения: try/except CSV

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd

def load_listings(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except FileNotFoundError as e:
        raise FileNotFoundError(
            f'Файл не найден: {path}. Положите listings.csv рядом с ноутбуком.'
        ) from e


def clean_listings(raw: pd.DataFrame) -> pd.DataFrame:
    return raw.dropna(subset=['accommodates', 'price']).copy()


def assert_usable(frame: pd.DataFrame) -> None:
    if len(frame) == 0:
        raise ValueError('пустой DataFrame после очистки')
    for col in ('accommodates', 'price'):
        if col not in frame.columns:
            raise ValueError(f'нет столбца {col}')


## Урок. 1. Успешная загрузка

In [ ]:
LISTINGS_PATH = None
for p in (Path('listings.csv'), Path('../../data/listings.csv'), Path('../data/listings.csv')):
    if p.exists():
        LISTINGS_PATH = p.resolve()
        break
assert LISTINGS_PATH is not None
df = load_listings(LISTINGS_PATH)
print(df.shape)

## Урок. 2. Ошибка пути

In [ ]:
caught = False
try:
    load_listings(Path('no_such_file.csv'))
except FileNotFoundError as e:
    caught = True
    print('Поймали:', e)
assert caught
ERROR_MSG_NOTE = (
    'Сообщение должно содержать путь и подсказку: '
    'положите listings.csv рядом с ноутбуком'
)
print(ERROR_MSG_NOTE)

## Урок. 3. Очистка

In [ ]:
raw = df.copy()
raw.loc[raw.index[0], 'price'] = None
cleaned = clean_listings(raw)
assert len(cleaned) == len(raw) - 1
print(len(raw), len(cleaned))

## Урок. 4. assert_usable

In [ ]:
empty = pd.DataFrame(columns=['accommodates', 'price'])
try:
    assert_usable(empty)
except ValueError as e:
    print('empty:', e)
bad = pd.DataFrame({'accommodates': [1.0, 2.0], 'number_of_reviews': [8, 9]})
try:
    assert_usable(bad)
except ValueError as e:
    print('bad cols:', e)

## Урок. 5. Pipeline

In [ ]:
pipeline_df = clean_listings(load_listings(LISTINGS_PATH))
assert_usable(pipeline_df)
print('pipeline', pipeline_df.shape)

## Урок. 6. Checkpoint

In [ ]:
WHY_TRY_FIRST = (
    'Ошибка файла должна быть понятной до fit: иначе traceback sklearn '
    'маскирует проблему данных.'
)
print(WHY_TRY_FIRST)

## ДЗ. 1–2. safe_load

In [ ]:
def safe_load(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        return None


assert safe_load(Path('no_such_file.csv')) is None
ok = safe_load(LISTINGS_PATH)
assert ok is not None and len(ok) > 0
print(ok.shape)

## ДЗ. 3. neighbourhood NaN

In [ ]:
sample = pd.DataFrame({
    'accommodates': [1.0, 2.0],
    'price': [10.0, 20.0],
    'neighbourhood': [None, 'center'],
})
print(clean_listings(sample))

## ДЗ. 4. Пустой после clean

In [ ]:
tiny = pd.DataFrame({'accommodates': [1.0], 'price': [None]})
cleaned = clean_listings(tiny)
status = 'empty after clean' if len(cleaned) == 0 else 'ok'
print(status)